# ДЗ — Fine-tune за полчаса: учим крошечную модель выдуманному факту (LoRA / QLoRA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITrubnikov/Train_of_Thought-homework/blob/main/notebooks/module-7-7-finetuning/notebook.ipynb)

> Если закрыли лекцию: это [Модуль 7.7](https://itrubnikov.github.io/Train_of_Thought/docs/modules/07-7-finetuning/) курса «От нуля до своих агентов».

За ~30 минут вы дообучаете готовую модель `Qwen2.5-0.5B-Instruct` так, чтобы она
выучила факты про **вымышленное** устройство «Кьюби». Его нет в интернете и не
было в обучающих данных модели — поэтому любой верный ответ после файнтюна это
**доказуемо выученное из ваших данных**, а не «модель и так знала». Это и делает
выводы наглядными.

**Что вы увидите тремя глазами:**
1. **Знание появилось** — до файнтюна база на «Кто создал Кьюби?» придумывает
   ерунду; после — отвечает по нашим фактам.
2. **Обобщает, а не зубрит** — отвечает верно на перефразировки, которых не было
   в обучении.
3. **Не сломалась** — на обычных вопросах (2+2, перевод слова) ведёт себя как
   раньше. Это держится за счёт rehearsal — об этом ниже.

**Главный урок:** LoRA меняет ~1–2% весов, адаптер весит десятки МБ. Но если
учить агрессивно на узких данных — модель забывает всё остальное (catastrophic
forgetting). Лечится подмешиванием обычных примеров (rehearsal) и эваллами.

**Что нужно:** Colab с GPU. `Runtime → Change runtime type → T4 GPU`.
Запустите ноутбук целиком (`Runtime → Run all`) — правок не требуется.

## Шаг 0. Среда и GPU

In [ ]:
# Версии запиннены под проверенный стек (июнь 2026). torch берём из Colab.
!pip install -q "transformers==5.12.1" "trl==1.6.0" "peft==0.19.1" "datasets==5.0.0" "accelerate==1.14.0" "bitsandbytes==0.49.2"

import os, json, glob, gc, urllib.request
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE != "cuda":
    print("ВНИМАНИЕ: CUDA-GPU не найден. LoRA-часть пойдёт (на CPU/MPS медленнее),")
    print("а QLoRA-секция (4-bit) требует CUDA и будет аккуратно пропущена.")
    print("На Colab включите GPU: Runtime -> Change runtime type -> T4 GPU.")

## Шаг 1. Вымышленное знание: датасет про «Кьюби»

«Кьюби» — выдуманный карманный ИИ-помощник (его придумали для этого урока).
В обучающих данных модели его нет, проверить факты в интернете нельзя — значит,
если после файнтюна модель отвечает про него верно, она выучила это **только**
из нашего датасета.

Датасет из двух частей:
- **knowledge** — факты про Кьюби, по нескольку перефразировок на факт (учим сам
  факт, а не конкретную строку);
- **rehearsal** — обычные вопросы (математика, география…), которые модель и так
  знает. Они «держат» общие способности и не дают модели скатиться в режим
  «отвечаю всё про Кьюби». Это ключевой приём против catastrophic forgetting.

Формат — `messages` (system / user / assistant): `trl` сам применит chat template.

In [ ]:
RAW = ("https://raw.githubusercontent.com/ITrubnikov/Train_of_Thought-homework/"
       "main/notebooks/module-7-7-finetuning/data")
os.makedirs("data", exist_ok=True)

def fetch(name):
    local = os.path.join("data", name)
    if not os.path.exists(local):
        urllib.request.urlretrieve(f"{RAW}/{name}", local)
    return local

train_path = fetch("qubi_train.jsonl")
held_path  = fetch("qubi_eval.jsonl")
train_rows = [json.loads(l) for l in open(train_path, encoding="utf-8")]
held_rows  = [json.loads(l) for l in open(held_path, encoding="utf-8")]
SYSTEM = train_rows[0]["messages"][0]["content"]

n_reh = sum(1 for r in train_rows if r.get("type") == "rehearsal")
print(f"train: {len(train_rows)} ({len(train_rows)-n_reh} про Кьюби + {n_reh} rehearsal)")
print(f"held:  {len(held_rows)} (перефразировки + контрольные вопросы)")
print("\nпример обучающего примера:")
print(json.dumps(train_rows[0]["messages"], ensure_ascii=False, indent=2))

## Шаг 2. Базовая модель и честный замер «ДО»

Загружаем `Qwen2.5-0.5B-Instruct` (0.5 млрд параметров — крохотная по меркам LLM)
и спрашиваем про Кьюби. Один и тот же системный промпт будем использовать
везде — и в замере «до», и в обучении, и в замере «после», — чтобы единственной
переменной были веса, а не формулировка.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE)

@torch.no_grad()
def ask(model, messages, max_new_tokens=64):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

facts = [r for r in held_rows if r.get("type") == "fact_paraphrase"]
def user_msgs(row):
    return [m for m in row["messages"] if m["role"] != "assistant"]

# Запоминаем ответы базы, чтобы потом сравнить «до/после».
before = {}
print("===== ДО файнтюна (база придумывает про Кьюби) =====")
for r in facts:
    q = user_msgs(r)[-1]["content"]
    before[q] = ask(base, user_msgs(r))
for r in facts[:5]:
    q = user_msgs(r)[-1]["content"]
    print("Q:", q, "\nA:", before[q], "\n")

## Шаг 3. LoRA — учим ~1–2% параметров

LoRA замораживает все веса базы `W` и добавляет к выбранным матрицам маленькую
поправку `W → W + B·A`, где `B` и `A` — узкие (low-rank) матрицы ранга `r`.
Обучаются только они. На печати ниже — сколько это в процентах от всех весов.

Используем `trl.SFTTrainer`: ему достаточно отдать датасет с `messages`,
токенайзер и `peft_config` — он сам применит chat template и обернёт модель в LoRA.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

train_ds = load_dataset("json", data_files=train_path, split="train")

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
args = SFTConfig(
    output_dir="qubi-lora",
    num_train_epochs=3,            # 3 эпохи: учит факты, но ещё не «перекручено»
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_length=384,
    report_to="none",
    fp16=(DEVICE == "cuda"),
)
trainer = SFTTrainer(model=base, args=args, train_dataset=train_ds,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()   # <-- вот насколько мало мы учим
trainer.train()

## Шаг 4. Адаптер — крошечный артефакт

Результат обучения — не новая модель на гигабайты, а **адаптер**: только матрицы
`A` и `B`. Сохраняем и смотрим размер.

In [ ]:
ADAPTER = "qubi-lora-adapter"
trainer.model.save_pretrained(ADAPTER)

adapter_mb = sum(os.path.getsize(p) for p in glob.glob(ADAPTER + "/*") if os.path.isfile(p)) / 1e6
base_gb = sum(p.numel() for p in base.parameters()) * 2 / 1e9   # ~2 байта/параметр в fp16
print(f"адаптер на диске: {adapter_mb:.1f} МБ")
print(f"для сравнения, веса базы ~{base_gb:.1f} ГБ")
print("=> адаптер можно носить с собой и навешивать на базу когда нужно.")

## Шаг 5. Замер «ПОСЛЕ» — навешиваем адаптер на чистую базу

Подводный камень: **не генерируйте сразу из `trainer.model`** — после `train()`
модель осталась в обучающем режиме (dropout, выключенный кэш, выровненный конфиг),
и `generate` выдаёт мусор. Правильно — как в проде: грузим чистую базу и
подключаем сохранённый адаптер.

In [ ]:
from peft import PeftModel

del trainer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

clean = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE)
ft = PeftModel.from_pretrained(clean, ADAPTER)
ft.train(False)   # режим инференса, не обучения

print("===== ДО / ПОСЛЕ на held-out перефразировках (их не было в обучении) =====\n")
for r in facts:
    q = user_msgs(r)[-1]["content"]
    print("Q     :", q)
    print("ДО    :", before[q])
    print("ПОСЛЕ :", ask(ft, user_msgs(r)), "\n")

print("===== Контроль: общие вопросы НЕ должны превратиться в «про Кьюби» =====")
for r in [r for r in held_rows if r.get("type") == "control"]:
    print("Q:", r["messages"][-1]["content"], "->", ask(ft, r["messages"], max_new_tokens=40))

## Что мы увидели — три вывода

1. **Знание появилось.** До файнтюна модель про Кьюби фантазировала; после —
   отвечает по нашим фактам (создатель, год, девиз, из чего сделан…).
2. **Обобщает.** Верные ответы идут на перефразировки, которых **не было** в
   обучении, — значит модель выучила факт, а не конкретную строку.
3. **Не сломалась.** Математика и переводы по-прежнему работают — потому что мы
   подмешали rehearsal. Уберите его (или поднимите число эпох) — и модель начнёт
   отвечать «про Кьюби» даже на «2+2». Это catastrophic forgetting вживую.

Заметили, что 1–2 ответа всё же путаются (например, цена ↔ память)? Крошечная
модель не идеальна. Понять, стало ли в среднем лучше, можно только измерением —
это эваллы (модуль 9). «На глаз» — не метод.

## Шаг 6. QLoRA: тот же приём, но модель 3B влезает в 4-bit

LoRA не уменьшает базу — `Qwen2.5-3B` в fp16 это ~6 ГБ, и на бесплатном T4 (16 ГБ)
вместе с активациями обучать её тяжело. **QLoRA** грузит базу в **4-bit** (~2 ГБ),
а сверху обучает те же LoRA-адаптеры. Ниже — реально грузим 3B в 4-bit и
дообучаем на том же датасете. Секция требует CUDA, на CPU/MPS — пропускается.

In [ ]:
if DEVICE != "cuda":
    print("QLoRA-секция требует CUDA-GPU (например, Colab T4). На", DEVICE, "— пропускаем.")
else:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel
    from trl import SFTConfig, SFTTrainer

    BIG = "Qwen/Qwen2.5-3B-Instruct"
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    big_tok = AutoTokenizer.from_pretrained(BIG)
    if big_tok.pad_token is None:
        big_tok.pad_token = big_tok.eos_token

    big = AutoModelForCausalLM.from_pretrained(BIG, quantization_config=bnb, device_map="auto")
    print(f"3B загружена в 4-bit, занято на GPU: {torch.cuda.memory_allocated()/1e9:.2f} ГБ "
          f"(в fp16 та же модель весила бы ~6 ГБ)")
    big = prepare_model_for_kbit_training(big)

    q = "Кто создал Кьюби? Ответь одним предложением."
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]

    @torch.no_grad()
    def ask_big(model):
        prompt = big_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = big_tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inp, max_new_tokens=48, do_sample=False,
                             pad_token_id=big_tok.pad_token_id)
        return big_tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    print("3B ДО QLoRA:", ask_big(big))

    qlora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    qargs = SFTConfig(
        output_dir="qubi-qlora",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        max_length=384,
        report_to="none",
        fp16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},   # нужно для PEFT + checkpointing
    )
    qtr = SFTTrainer(model=big, args=qargs, train_dataset=train_ds,
                     processing_class=big_tok, peft_config=qlora)
    qtr.model.print_trainable_parameters()
    qtr.train()
    qtr.model.save_pretrained("qubi-qlora-adapter")

    del qtr, big
    gc.collect(); torch.cuda.empty_cache()

    big2 = AutoModelForCausalLM.from_pretrained(BIG, quantization_config=bnb, device_map="auto")
    big_ft = PeftModel.from_pretrained(big2, "qubi-qlora-adapter")
    big_ft.train(False)
    print("3B ПОСЛЕ QLoRA:", ask_big(big_ft))

## (Опционально) Выложить адаптер на HuggingFace Hub

Адаптер — это файл, которым можно поделиться. Чтобы выложить, добавьте токен:
в Colab — `Secrets` (значок ключа слева) → `HF_TOKEN`. Без токена ячейка просто
пропускается и `Run all` не падает.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    print("HF_TOKEN не найден -> публикацию пропускаем (это нормально).")
    print("Чтобы выложить: Colab -> Secrets -> HF_TOKEN, затем перезапустите ячейку.")
else:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    REPO = "qubi-lora-0_5b"   # поменяйте на свой: <ваш-ник>/qubi-lora
    ft.push_to_hub(REPO)
    tok.push_to_hub(REPO)
    print("выложено в", REPO, "— теперь адаптер можно подключать через peft из любого места")

## Подводные камни (почему «не работает»)

- **Chat template.** Базовая и инструкт-модель ждут разный формат. Мы отдаём
  `messages` и даём `trl` применить шаблон — не склеиваем строки руками.
- **pad / eos.** У Qwen ход ассистента заканчивается на `<|im_end|>`. Если
  токенайзер без `pad_token` — ставим `pad_token = eos_token`.
- **Не генерируйте из `trainer.model`.** После обучения модель в train-режиме —
  `generate` ломается. Сохраните адаптер и подключите к чистой базе.
- **OOM на больших моделях.** fp16-веса не влезают → QLoRA (4-bit) +
  `gradient_checkpointing` + `paged_adamw_8bit`.
- **Переобучение и забывание.** Узкие данные + много эпох = модель отвечает всё
  «про Кьюби». Лечение: меньше эпох/lr, rehearsal, и обязательно эваллы.
- **target_modules.** Для фактов берём и attention, и MLP-проекции; для одного
  стиля иногда хватает `q_proj`/`v_proj`.

## Задачи

Запускается всё выше без правок. Дальше — ваша часть (расширения, не обязательны
для «работает»):

1. **Свой факт-сет.** Замените «Кьюби» на свою выдуманную сущность: отредактируйте
   `build_dataset.py` (или соберите `qubi_train.jsonl` руками) и дообучите. Покажите
   before/after на 5 перефразировках.
2. **Поймайте forgetting.** Уберите rehearsal-примеры (или поставьте
   `num_train_epochs=8`) и покажите, как контрольные вопросы «съезжают» в Кьюби.
3. **Поиграйте LoRA.** Поменяйте `r`, `lora_alpha`, `target_modules` и опишите, как
   меняются размер адаптера и качество ответов.
4. **(Опц.) Выложите адаптер** на HF Hub и подключите его к базе в свежей сессии.

Артефакт — публичная ссылка на ваш ноутбук (Colab/Kaggle) в чат как
`[Модуль 7.7, ДЗ N] {ссылка}`.

## Что дальше

- [Модуль 8 — Что такое агент](https://itrubnikov.github.io/Train_of_Thought/docs/modules/08-what-is-agent/).
- Быстрее и экономнее: [Unsloth](https://github.com/unslothai/unsloth),
  [HuggingFace smol-course](https://github.com/huggingface/smol-course).
- Измерить, помог ли файнтюн: модуль про эваллы.

---

Лицензия: учебные материалы курса «От нуля до своих агентов». Датасет про «Кьюби» —
вымышленный, придуман для урока.